# TRACE-ESUS — heart-failure redundancy grid

**Endotype discovery** and **model comparison** across the full renal x heart-failure grid.

Run top to bottom. Cell 2 sets scale; everything after it regenerates from saved CSVs,
so figures can be re-styled without re-running any model.

### What changed from the locked experiments

1. **Markers renamed** to NT-proBNP / PTFV1 / competing-vascular (legacy enum aliases retained).
2. **Heart failure added as a second nuisance** on PTFV1: `B_PTFV1 = lambda[Z,1] + gamma_H * H + eps`.
   Prevalence renal 30%, HF 7%. HF is drawn *after* every historical draw, so `gamma_H = 0`
   reproduces the locked outputs bit-for-bit.
3. **Models generalized** to a nuisance design matrix. Previously only renal -> NT-proBNP was
   representable, which made the counterfactual query algebraically identical to the posterior
   regardless of the data. The biology mask is now renal -> NT-proBNP, HF -> PTFV1.
4. **Four prespecified profiles** reported everywhere: uncomplicated / renal-only / HF-only /
   redundant. False-atrial attribution within a profile uses that profile's
   competing-mechanism patients as the denominator.

Nothing here writes to a proposal-cited directory. Seed roots are salted and disjoint.

In [1]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd()))
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np, pandas as pd
from configs.endotype_discovery import CONFIG as ENDOTYPE_CONFIG
from configs.model_comparison import CONFIG as COMPARISON_CONFIG
from traceesus.experiments import hf_grid
from traceesus.experiments.endotype_discovery import multi_nuisance as mn
from traceesus.plotting import hf_panels as panels

OUT = pathlib.Path("outputs_hf_grid"); OUT.mkdir(exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 1.26.4 | pandas 2.2.2


## 1. Scale

`REPEATS = 20` is a few minutes and is enough to see every effect.
Use `200` for the figures you actually present.

In [2]:
REPEATS = 20
WORKERS = 4
RENAL_LEVELS = (0.0, 0.5, 1.0, 1.5)
HF_LEVELS = (0.0, 0.5, 1.0, 1.5)

print(f"{len(RENAL_LEVELS)*len(HF_LEVELS)} cells x {REPEATS} repeats")

16 cells x 20 repeats


## 2. Endotype discovery — unsupervised, 4x4 grid

Five models per repeat, all on identical cohorts with per-model fit streams:

| Model | Nuisance paths | Query |
|---|---|---|
| Associative LCM | none | posterior |
| Renal-adjusted LCM | renal, free | posterior |
| Two-nuisance adjusted LCM | renal + HF, free on all markers | posterior |
| Two-path causal SCM | renal -> NT-proBNP, HF -> PTFV1 | posterior |
| Two-path causal SCM | *identical fit* | **counterfactual** |

The last two rows share a fit, so any difference is attributable to the query alone.

In [3]:
%%time
latent = hf_grid.run_latent_grid(
    ENDOTYPE_CONFIG,
    renal_levels=RENAL_LEVELS, hf_levels=HF_LEVELS,
    repeats=REPEATS, workers=WORKERS,
)
latent.to_csv(OUT / "latent_grid_raw.csv", index=False)
print(latent.shape, "|", latent.method.nunique(), "models")
latent.head(3)

(1600, 32) | 5 models
CPU times: user 157 ms, sys: 126 ms, total: 283 ms
Wall time: 42.3 s


,repeat,renal_effect_sd,heart_failure_effect_sd,method,accuracy,adjusted_rand_index,false_atrial_renal_competing,brier_score,expected_calibration_error,mean_posterior_entropy,...,accuracy__heart_failure_only,false_atrial__heart_failure_only,mean_posterior_entropy__heart_failure_only,subgroup_size__heart_failure_only,competing_subgroup_size__heart_failure_only,accuracy__redundant,false_atrial__redundant,mean_posterior_entropy__redundant,subgroup_size__redundant,competing_subgroup_size__redundant
0,0,0.0,0.0,Associative latent class model,0.793,0.342736,0.129496,0.144884,0.076266,0.372239,...,0.800000,0.107143,0.359381,60,28,0.909091,0.076923,0.345141,22,13
1,0,0.0,0.0,Renal-adjusted associative latent class model,0.793,0.342736,0.194245,0.146857,0.085198,0.368300,...,0.766667,0.107143,0.354232,60,28,0.863636,0.153846,0.351746,22,13
2,0,0.0,0.0,Two-nuisance adjusted associative latent model,0.794,0.345087,0.194245,0.146879,0.085820,0.368130,...,0.800000,0.107143,0.355329,60,28,0.863636,0.153846,0.354139,22,13


In [4]:
# Headline cell: strong renal AND strong heart failure
cell = latent[(latent.renal_effect_sd == 1.5) & (latent.heart_failure_effect_sd == 1.5)]
cell.groupby("method")[[
    "accuracy", "false_atrial_renal_competing",
    "false_atrial__uncomplicated", "false_atrial__renal_only",
    "false_atrial__heart_failure_only", "false_atrial__redundant",
]].mean().round(3)

,accuracy,false_atrial_renal_competing,false_atrial__uncomplicated,false_atrial__renal_only,false_atrial__heart_failure_only,false_atrial__redundant
method,,,,,,
Associative latent class model,0.558,0.775,0.160,0.772,0.176,0.815
Renal-adjusted associative latent class model,0.800,0.187,0.175,0.172,0.402,0.376
Two-nuisance adjusted associative latent model,0.806,0.188,0.188,0.191,0.187,0.157
Two-path biologically constrained latent SCM,0.814,0.178,0.175,0.179,0.189,0.183
Two-path latent SCM (counterfactual query),0.817,0.172,0.177,0.174,0.187,0.162


## 3. Model comparison — supervised, same grid

Same generator, but labels are used in training. This is supervised classification,
not endotype discovery, and the boundary is kept explicit.

In [5]:
%%time
supervised = hf_grid.run_supervised_grid(
    ENDOTYPE_CONFIG.simulation, COMPARISON_CONFIG,
    master_seed=ENDOTYPE_CONFIG.master_seed,
    renal_levels=RENAL_LEVELS, hf_levels=HF_LEVELS,
    repeats=REPEATS, workers=WORKERS,
)
supervised.to_csv(OUT / "supervised_grid_raw.csv", index=False)
print(supervised.shape)

cell_s = supervised[(supervised.renal_effect_sd == 1.5) & (supervised.heart_failure_effect_sd == 1.5)]
cell_s.groupby("method")[[
    "accuracy", "false_atrial__uncomplicated", "false_atrial__renal_only",
    "false_atrial__heart_failure_only", "false_atrial__redundant",
]].mean().round(3)

(1600, 32)
CPU times: user 166 ms, sys: 106 ms, total: 272 ms
Wall time: 42.7 s


,accuracy,false_atrial__uncomplicated,false_atrial__renal_only,false_atrial__heart_failure_only,false_atrial__redundant
method,,,,,
Logistic regression (+ kidney + heart failure),0.828,0.173,0.167,0.193,0.167
Logistic regression (+ kidney status),0.824,0.161,0.156,0.400,0.319
Logistic regression (biomarkers only),0.806,0.112,0.336,0.357,0.583
Supervised two-path SCM (counterfactual query),0.828,0.171,0.167,0.187,0.163
Supervised two-path SCM (posterior),0.827,0.173,0.168,0.187,0.158


## 4. Panel A — the problem, at patient level

One labelled cohort per renal level, drawn outside every repeat loop from its own
salted root. This never enters a summarized estimate; it exists so the failure can be
seen rather than asserted.

**Read the two rows against each other.** The top row is the truth. The bottom row is
the identical set of points coloured by what an unadjusted latent class model actually
returns. Open circles are renally impaired patients in both rows.

The naive expectation — that renal distortion blurs the two mechanism clouds together —
is wrong, and the figure says so: best-case separation in this plane holds at ~0.77
across all three facets. Renal distortion shifts the impaired patients of *both*
mechanisms by the same amount, so it does not destroy the mechanism signal.

What it does is make kidney status the loudest axis of variation in the plane. An
unsupervised model follows variance, so its split rotates off the mechanism contrast and
onto the renal contrast. This is the mechanism behind the K=1 null result: the model is
not failing to find structure, it is finding the wrong structure confidently.


In [6]:
SCATTER_HF_SD = 1.5   # heart-failure path held on while renal distortion varies

scatter = hf_grid.cohort_scatter_sample(
    ENDOTYPE_CONFIG.simulation, ENDOTYPE_CONFIG.fitting,
    master_seed=ENDOTYPE_CONFIG.master_seed,
    heart_failure_effect_sd=SCATTER_HF_SD, patients=4_000,
)
scatter.to_csv(OUT / "cohort_scatter_sample.csv", index=False)

panels.plot_mechanism_scatter(
    scatter, FIG / "A_marker_plane",
    subtitle=(
        f"Heart failure fixed at {SCATTER_HF_SD:g} SD on PTFV1 (7% prevalence); "
        "4,000 patients per facet"
    ),
    source="outputs_hf_grid/cohort_scatter_sample.csv",
)

# The numbers printed on the figure, in a form that can be pasted into a caption.
import numpy as np
for level, block in scatter.groupby("renal_effect_sd"):
    found = (block.discovered_class == "class A").to_numpy()
    agree = lambda a, b: max(float(np.mean(a == b)), 1.0 - float(np.mean(a == b)))
    print(
        f"renal {level:<5g} "
        f"best 2-marker separation {panels._gaussian_separability(block[['nt_probnp','ptfv1']].to_numpy(float), block.true_mechanism.to_numpy()):.3f}"
        f" | discovered split matches mechanism {agree(found, block.true_mechanism.to_numpy() == 'atrial'):.3f}"
        f" | matches kidney status {agree(found, block.renal_dysfunction.to_numpy() == 1):.3f}"
    )


renal 0     best 2-marker separation 0.768 | discovered split matches mechanism 0.817 | matches kidney status 0.528
renal 0.75  best 2-marker separation 0.776 | discovered split matches mechanism 0.800 | matches kidney status 0.599
renal 1.5   best 2-marker separation 0.760 | discovered split matches mechanism 0.554 | matches kidney status 0.937


## 5. Panel A2 — the drift, measured

Panel A shows the wrong split at three snapshots. This measures it continuously.

A latent class has no intrinsic meaning; it is whatever the likelihood rewards. The
unadjusted model factorizes as **p(Z) p(R | Z) p(B | Z)** — kidney status is modelled as
something the class *causes*. The true edge is the opposite, R -> B, and the model has no
slot for it, so the only place it can put a bimodal NT-proBNP is inside the class itself.

Panel a tracks whether the discovered split corresponds to the mechanism contrast or to
kidney status. Panel b opens up the class the model would read as atrial-like and shows
who is actually in it. The model never stops being confident; it stops being about
mechanism.

The renal sweep runs to 2.0 SD, past the locked grid's 1.5, so the asymptote is visible.
Its seed root is salted and independent of every cited artifact.


In [7]:
DRIFT_REPEATS = 40

drift = hf_grid.identity_drift_sweep(
    ENDOTYPE_CONFIG.simulation, ENDOTYPE_CONFIG.fitting,
    master_seed=ENDOTYPE_CONFIG.master_seed,
    heart_failure_effect_sd=SCATTER_HF_SD,
    repeats=DRIFT_REPEATS, workers=WORKERS,
)
drift.to_csv(OUT / "identity_drift_raw.csv", index=False)

panels.plot_identity_drift(
    drift, FIG / "A2_identity_drift",
    subtitle=(
        f"Unadjusted associative LCM, n = {ENDOTYPE_CONFIG.simulation.training_patients} "
        f"per fit, {DRIFT_REPEATS} repeats per level; HF fixed at {SCATTER_HF_SD:g} SD. "
        "Bands are 95% t intervals."
    ),
    source="outputs_hf_grid/identity_drift_raw.csv",
)

drift.groupby("renal_effect_sd")[[
    "agreement_with_mechanism", "agreement_with_kidney_status",
    "composition__atrial_normal_kidneys", "composition__competing_impaired_kidneys",
]].mean().mul(100).round(1)


,agreement_with_mechanism,agreement_with_kidney_status,composition__atrial_normal_kidneys,composition__competing_impaired_kidneys
renal_effect_sd,,,,
0.00,81.6,52.5,59.0,4.9
0.25,81.2,54.4,55.5,7.3
0.50,79.8,57.8,52.4,10.7
0.75,77.1,63.6,48.2,14.9
1.00,72.5,70.9,42.6,20.6
1.25,65.5,81.3,32.1,29.2
1.50,57.2,91.8,15.3,39.9
1.75,52.6,98.0,4.2,46.7
2.00,51.7,99.6,0.9,50.2


## 6. Panels B-F — aggregate results

Accent colour marks the method under test; baselines are grey. Every figure carries its
source file, writes PNG + PDF, and reduces repeats with t-based 95% intervals.


In [8]:
LATENT_ACCENT = {
    mn.CAUSAL_TWO_NUISANCE: panels.ACCENT,
    mn.COUNTERFACTUAL_TWO_NUISANCE: panels.ACCENT_CF,
}
SRC_L = "outputs_hf_grid/latent_grid_raw.csv"
SRC_S = "outputs_hf_grid/supervised_grid_raw.csv"

# B. Heat maps — false atrial in the redundant profile across the whole grid
panels.plot_false_atrial_heatmaps(
    latent,
    ["Associative latent class model",
     "Renal-adjusted associative latent class model",
     mn.CAUSAL_TWO_NUISANCE],
    FIG / "B_latent_redundant_heatmap",
    title="Endotype discovery: false atrial attribution, redundant profile",
    source=SRC_L,
)

# C. Subgroup bars at the hardest cell
panels.plot_subgroup_bars(
    latent, FIG / "C_latent_subgroup_bars",
    renal_effect_sd=1.5, hf_effect_sd=1.5,
    accent_methods=LATENT_ACCENT,
    title="Endotype discovery by nuisance profile (renal 1.5 SD, HF 1.5 SD)",
    source=SRC_L,
)

# D. HF slice at fixed strong renal distortion
panels.plot_hf_slice(
    latent, FIG / "D_latent_hf_slice",
    renal_effect_sd=1.5, accent_methods=LATENT_ACCENT,
    title="Rising heart-failure contamination of PTFV1 (renal fixed at 1.5 SD)",
    source=SRC_L,
)

# E. Does the query diverge from the posterior on the same fitted model?
panels.plot_query_divergence(
    latent, FIG / "E_query_divergence",
    posterior_method=mn.CAUSAL_TWO_NUISANCE,
    counterfactual_method=mn.COUNTERFACTUAL_TWO_NUISANCE,
    renal_effect_sd=1.5, source=SRC_L,
)

# F. Supervised comparison, same profiles
panels.plot_subgroup_bars(
    supervised, FIG / "F_supervised_subgroup_bars",
    renal_effect_sd=1.5, hf_effect_sd=1.5,
    accent_methods={
        hf_grid.SUPERVISED_SCM_TWO_PATH: panels.ACCENT,
        hf_grid.SUPERVISED_SCM_COUNTERFACTUAL: panels.ACCENT_CF,
    },
    title="Model comparison by nuisance profile (renal 1.5 SD, HF 1.5 SD)",
    source=SRC_S,
)

print("figures written to", FIG)
sorted(p.name for p in FIG.glob("*.png"))

figures written to outputs_hf_grid/figures


['A2_identity_drift.png',
 'A_marker_plane.png',
 'B_latent_redundant_heatmap.png',
 'C_latent_subgroup_bars.png',
 'D_latent_hf_slice.png',
 'E_query_divergence.png',
 'F_supervised_subgroup_bars.png',
 'figure2_recovery_lines.png',
 'figure2_recovery_lines_renal_denominator.png']

## 7. Manuscript Figure 2 replacement

Direct successor to the locked Figure 2: same axes, same level labels, same two panels —
but with the heart-failure path held on and the two adjustment tiers the single-nuisance
figure could not represent. The counterfactual query is excluded; this figure is about
what the model *represents*, not how it is queried.

Two versions are written because panel b's denominator changes the conclusion:

| File | Panel b denominator | What it shows |
|---|---|---|
| `figure2_recovery_lines` | redundant profile (renal **and** HF) | renal-adjusted model breaks at 37.6% — the new result |
| `figure2_recovery_lines_renal_denominator` | renal/competing subgroup | the locked Figure 2 metric, for continuity |

Use the first. Keep the second so a reviewer comparing against the published figure can
see that the original metric was not quietly swapped for a friendlier one.


In [9]:
FIG2_CAPTION = (
    f"Unlabeled training n = {ENDOTYPE_CONFIG.simulation.training_patients}; "
    f"independent test n = {ENDOTYPE_CONFIG.simulation.test_patients:,}; "
    f"{REPEATS} paired repeats per cell; heart-failure path fixed at 1.5 SD on PTFV1 "
    f"({ENDOTYPE_CONFIG.simulation.heart_failure_prevalence:.0%} prevalence). "
    "Bars are 95% t intervals."
)

# Primary: panel b on the redundant profile, where the renal-adjusted model fails.
panels.plot_recovery_lines(
    latent, FIG / "figure2_recovery_lines",
    hf_effect_sd=1.5,
    false_atrial_metric="false_atrial__redundant",
    panel_b_title="False atrial calls, redundant subgroup (renal + HF)",
    caption=FIG2_CAPTION, source=SRC_L,
)

# Continuity: panel b on the locked figure's own denominator.
panels.plot_recovery_lines(
    latent, FIG / "figure2_recovery_lines_renal_denominator",
    hf_effect_sd=1.5,
    false_atrial_metric="false_atrial_renal_competing",
    panel_b_title="False atrial calls, renal/competing subgroup",
    caption=FIG2_CAPTION, source=SRC_L,
)

# Plotted values, for the caption and the manuscript text.
fig2 = (
    latent[(latent.heart_failure_effect_sd == 1.5)
           & (latent.method != mn.COUNTERFACTUAL_TWO_NUISANCE)]
    .groupby(["method", "renal_effect_sd"])[
        ["accuracy", "false_atrial__redundant", "false_atrial_renal_competing"]
    ].mean().mul(100).round(1)
)
fig2


accuracy  \
method                                         renal_effect_sd             
Associative latent class model                 0.0                  81.6   
                                               0.5                  80.0   
                                               1.0                  72.7   
                                               1.5                  55.8   
Renal-adjusted associative latent class model  0.0                  81.6   
                                               0.5                  81.6   
                                               1.0                  81.2   
                                               1.5                  80.0   
Two-nuisance adjusted associative latent model 0.0                  82.0   
                                               0.5                  82.0   
                                               1.0                  81.8   
                                               1.5                  80.6   
Two-path biologically constrained latent SCM   0.0                  82.2   
                                               0.5                  82.5   
                                               1.0                  82.1   
                                               1.5                  81.4   

                                                                false_atrial__redundant  \
method                                         renal_effect_sd                            
Associative latent class model                 0.0                                 35.4   
                                               0.5                                 54.8   
                                               1.0                                 73.8   
                                               1.5                                 81.5   
Renal-adjusted associative latent class model  0.0                                 37.2   
                                               0.5                                 40.0   
                                               1.0                                 39.4   
                                               1.5                                 37.6   
Two-nuisance adjusted associative latent model 0.0                                 18.2   
                                               0.5                                 15.5   
                                               1.0                                 18.2   
                                               1.5                                 15.7   
Two-path biologically constrained latent SCM   0.0                                 17.5   
                                               0.5                                 17.5   
                                               1.0                                 15.9   
                                               1.5                                 18.3   

                                                                false_atrial_renal_competing  
method                                         renal_effect_sd                                
Associative latent class model                 0.0                                      17.1  
                                               0.5                                      32.4  
                                               1.0                                      60.9  
                                               1.5                                      77.5  
Renal-adjusted associative latent class model  0.0                                      18.1  
                                               0.5                                      17.4  
                                               1.0                                      18.3  
                                               1.5                                      18.7  
Two-nuisance adjusted associative latent model 0.0                                      17.5  
           

## 8. Subgroup adequacy

Renal 30% x HF 7% makes the redundant cell about 2% of patients, and only its
competing-mechanism half enters the false-atrial denominator. Check the size before
quoting a redundant-cell number.

In [10]:
sizes = latent.groupby(["renal_effect_sd", "heart_failure_effect_sd"])[
    ["subgroup_size__redundant", "competing_subgroup_size__redundant"]
].median().astype(int)
print(sizes.to_string())
print("\nIf the competing cell is under ~30, raise test_patients or HF prevalence")
print("before presenting a redundant-profile estimate.")

                                         subgroup_size__redundant  competing_subgroup_size__redundant
renal_effect_sd heart_failure_effect_sd                                                              
0.0             0.0                                            21                                   9
                0.5                                            20                                  10
                1.0                                            23                                  11
                1.5                                            20                                   9
0.5             0.0                                            21                                  11
                0.5                                            21                                  10
                1.0                                            23                                  13
                1.5                                            20                 

## 9. What to read off these

**The signal is not lost — the split is.** Panel A: best-case separation in the
NT-proBNP / PTFV1 plane is flat across renal distortion, yet the unadjusted model's
split goes from matching mechanism to matching kidney status almost perfectly. The
problem is confident misattribution, not missing information. That is why adjustment
recovers so much, and why the K=1 null is the strongest result in the package.

**The nuisance structure has to match the biology.** At renal 1.5 / HF 1.5 the
renal-adjusted model — the one that looks correct under the old single-nuisance story —
still misattributes a large share of redundant patients, because heart failure is
contaminating PTFV1 and it has no term for it. Adding the second path fixes it.

**Adjustment, not causal labelling, is doing the work.** The two-nuisance *adjusted*
model and the two-path *causal* model land close together. The defensible claim stays
what it was: explicit nuisance-path modelling is what buys the gain.

**The query comparison is now a real measurement.** Under the one-path model,
sufficiency and disablement were monotone transforms of the posterior — identical by
algebra, whatever the data. With both paths representable, the disabled world retains
the patient's estimated background, so disablement is no longer a function of the
posterior and the two queries can disagree. Panel E shows whether they do.
